In [0]:
# ===================================================
# BLOCK 1 — CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
LANDING = f"{CATALOG}.simulation.simulated_hourly_equipment_operations_landing"
BRONZE = f"{CATALOG}.bronze.simulated_hourly_equipment_operations"
SILVER = f"{CATALOG}.silver.simulated_hourly_equipment_operations"
FACT_OEE = f"{CATALOG}.gold.fact_oee_hourly"
FACT_EVENTS = f"{CATALOG}.gold.fact_equipment_events_observed"
MART_DAILY = f"{CATALOG}.gold.mart_equipment_oee_daily"
MART_LOSSES = f"{CATALOG}.gold.mart_equipment_losses_daily"
EXPECTED_HOURLY_ROWS = 1051776
EXPECTED_EVENT_ROWS = 255639
EXPECTED_DAILY_ROWS = 43824
EXPECTED_SEED = 20260819

print("Day 4 operations validation configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — TABLE COUNTS AND DISCLOSURE (PYTHON)
# ===================================================

counts = [("LANDING", spark.table(LANDING).count()), ("BRONZE", spark.table(BRONZE).count()),
          ("SILVER", spark.table(SILVER).count()), ("FACT_OEE", spark.table(FACT_OEE).count()),
          ("FACT_EVENTS", spark.table(FACT_EVENTS).count()), ("MART_DAILY", spark.table(MART_DAILY).count()),
          ("MART_LOSSES", spark.table(MART_LOSSES).count())]
display(spark.createDataFrame(counts, ["dataset", "row_count"]))

count_map = dict(counts)
assert count_map["LANDING"] == count_map["BRONZE"] == count_map["SILVER"] == count_map["FACT_OEE"] == EXPECTED_HOURLY_ROWS
assert count_map["FACT_EVENTS"] == EXPECTED_EVENT_ROWS
assert count_map["MART_DAILY"] == count_map["MART_LOSSES"] == EXPECTED_DAILY_ROWS

assert spark.table(FACT_OEE).filter((~F.col("simulated_record_flag")) | (F.col("simulation_seed") != EXPECTED_SEED)).count() == 0
assert spark.table(FACT_EVENTS).filter(F.col("simulated_record_flag") | (F.col("record_origin") != "OBSERVED_SILVER_EQUIPMENT_EVENTS")).count() == 0
print("Counts and synthetic/observed disclosure passed.")

In [0]:
# ===================================================
# BLOCK 3 — TIME, OUTPUT AND KPI INVARIANTS (PYTHON)
# ===================================================

oee_df = spark.table(FACT_OEE)
invalid = oee_df.filter(
    (F.col("planned_production_time_seconds") != F.col("scheduled_time_seconds") - F.col("approved_planned_downtime_seconds"))
    | (F.col("operating_time_seconds") != F.col("planned_production_time_seconds") - F.col("unplanned_downtime_seconds"))
    | (F.col("operating_time_seconds") < 0)
    | (F.col("good_units") < 0) | (F.col("good_units") > F.col("total_units"))
    | (F.col("total_units") > F.col("theoretical_output_units") + 1e-9)
    | (F.col("availability") < 0) | (F.col("availability") > 1)
    | (F.col("utilization") < 0) | (F.col("utilization") > 1)
    | (F.col("quality") < 0) | (F.col("quality") > 1)
    | (F.col("oee") < 0) | (F.col("oee") > 1)
).count()
assert invalid == 0, f"OEE invariant failures: {invalid}"
print("Time, output and OEE KPI invariants passed.")

In [0]:
# ===================================================
# BLOCK 4 — OBSERVED EVENT GRAIN AND TIME VALIDATION (PYTHON)
# ===================================================

events_df = spark.table(FACT_EVENTS)
duplicate_events = events_df.groupBy("event_id").count().filter("count > 1").count()
invalid_event_times = events_df.filter(F.col("event_end_timestamp_utc") < F.col("event_timestamp_utc")).count()
unresolved_event_keys = events_df.filter(F.col("date_key").isNull() | F.col("site_key").isNull() | F.col("equipment_key").isNull()).count()
assert duplicate_events == 0
assert invalid_event_times == 0
assert unresolved_event_keys == 0
print("Observed event grain, time and keys passed.")

In [0]:
# ===================================================
# BLOCK 5 — CONTROLLED EVENT-BOUNDARY TESTS (PYTHON)
# ===================================================

boundary_rows = [
    ("UNPLANNED_DOWNTIME", 300, "SHORT_STOP"),
    ("UNPLANNED_DOWNTIME", 301, "DOWNTIME"),
    ("PLANNED_DOWNTIME", 3600, "PLANNED_DOWNTIME"),
    ("PLANNED_DOWNTIME", 3601, "PLANNED_OVERRUN"),
]
boundary_df = spark.createDataFrame(boundary_rows, ["event_type", "duration_seconds", "expected_classification"])
boundary_result_df = boundary_df.withColumn(
    "actual_classification",
    F.when((F.col("event_type") == "UNPLANNED_DOWNTIME") & (F.col("duration_seconds") > 300), "DOWNTIME")
     .when((F.col("event_type") == "UNPLANNED_DOWNTIME") & (F.col("duration_seconds") <= 300), "SHORT_STOP")
     .when((F.col("event_type") == "PLANNED_DOWNTIME") & (F.col("duration_seconds") > 3600), "PLANNED_OVERRUN")
     .otherwise("PLANNED_DOWNTIME"),
).withColumn("passed", F.col("actual_classification") == F.col("expected_classification"))
display(boundary_result_df)
assert boundary_result_df.filter(~F.col("passed")).count() == 0
print("Controlled 300/301 and 3600/3601 boundaries passed.")


In [0]:
# ===================================================
# BLOCK 6 — WEIGHTED SEVEN-DAY OEE VALIDATION (PYTHON)
# ===================================================

latest_date = spark.table(MART_DAILY).agg(F.max("production_date")).first()[0]
seven_day_df = spark.table(FACT_OEE).filter(
    F.col("production_date").between(F.date_sub(F.lit(latest_date), 6), F.lit(latest_date))
)
seven_day = (
    seven_day_df.agg(
        F.sum("planned_production_time_seconds").alias("planned"),
        F.sum("operating_time_seconds").alias("operating"),
        F.sum("theoretical_output_units").alias("theoretical"),
        F.sum("total_units").alias("total"),
        F.sum("good_units").alias("good"),
    )
    .withColumn("availability", F.col("operating") / F.col("planned"))
    .withColumn("utilization", F.col("total") / F.col("theoretical"))
    .withColumn("quality", F.col("good") / F.col("total"))
    .withColumn(
        "weighted_oee",
        F.col("availability") * F.col("utilization") * F.col("quality"),
    )
)
display(seven_day)
assert seven_day.filter((F.col("weighted_oee") < 0) | (F.col("weighted_oee") > 1)).count() == 0
print("Weighted seven-day OEE recomputation passed.")

In [0]:
# ===================================================
# BLOCK 7 — LATEST OBSERVED 24-HOUR WINDOW (PYTHON)
# ===================================================

latest_event_ts = events_df.agg(F.max("event_timestamp_utc")).first()[0]
latest_24h_df = events_df.filter(
    (F.col("event_timestamp_utc") > F.lit(latest_event_ts) - F.expr("INTERVAL 24 HOURS"))
    & (F.col("event_timestamp_utc") <= F.lit(latest_event_ts))
).groupBy("event_classification").agg(F.count("*").alias("event_count"), F.sum("duration_seconds").alias("duration_seconds"))
display(latest_24h_df.orderBy(F.desc("duration_seconds")))
assert latest_24h_df.count() > 0

In [0]:
# ===================================================
# BLOCK 8 — DETERMINISTIC SIGNATURE (PYTHON)
# ===================================================

signature_df = (
    spark.table(SILVER)
    .select(
        F.sha2(
            F.concat_ws(
                "|",
                "operation_record_id",
                "scheduled_time_seconds",
                "approved_planned_downtime_seconds",
                "unplanned_downtime_seconds",
                "total_units",
                "good_units",
                "simulation_seed",
            ),
            256,
        ).alias("row_signature"),
        F.col("total_units"),
        F.col("good_units"),
    )
    .agg(
        F.count("*").alias("row_count"),
        F.min("row_signature").alias("minimum_row_signature"),
        F.max("row_signature").alias("maximum_row_signature"),
        F.sum("total_units").alias("total_units"),
        F.sum("good_units").alias("good_units"),
    )
)

display(signature_df)

signature_result = signature_df.first()

assert signature_result["row_count"] == EXPECTED_HOURLY_ROWS
assert signature_result["total_units"] > 0
assert signature_result["good_units"] > 0
assert signature_result["good_units"] <= signature_result["total_units"]

print("Record all five values; they must match on the second run.")

In [0]:
# ===================================================
# BLOCK 9 — FINAL DAY 4 ACCEPTANCE (PYTHON)
# ===================================================

acceptance = {
    "hourly_oee_rows": EXPECTED_HOURLY_ROWS,
    "observed_event_rows": EXPECTED_EVENT_ROWS,
    "daily_oee_rows": EXPECTED_DAILY_ROWS,
    "simulation_seed": EXPECTED_SEED,
    "synthetic_oee_labeled": True,
    "observed_events_labeled": True,
    "boundary_tests": "PASSED",
    "weighted_seven_day_oee": "PASSED",
    "status": "PASSED",
}
display(spark.createDataFrame([acceptance]))
print("DAY 4 MODEL ACCEPTANCE: PASSED")
print("Synthetic standard OEE and observed equipment events remain separate.")